In [2]:
%load_ext autoreload
%autoreload 2

In [27]:
import pandas as pd
import numpy as np
import os
from tqdm import tqdm
import timm
import torchvision.transforms as T
from PIL import Image
import torch
from typing import List
from fungiclef.model import FungiMEEModel, FungiEnsembleModel
from fungiclef.dataset import EmbeddingMetadataDataset
from torch.utils.data import DataLoader

from sklearn.metrics import accuracy_score, f1_score, classification_report

from fungiclef.utils import get_poison_mapping

from PIL import Image

In [4]:
ckpt_path = "../checkpoints"
model_paths = ["dino_05241908",
"dino_05241702",
"dino_optuna_05241507",
"dino_optuna_05241449",
"dino_optuna_05241431",
"dino_optuna_05241354",
"dino_optuna_05241257",
"dino_optuna_05241222",
"dino_optuna_05241449",]

In [18]:
model_paths = ["dino_2_optuna_05250105",
"dino_2_optuna_05250045",
"dino_2_optuna_05250025",
"dino_2_optuna_05250006",
"dino_2_optuna_05242344",
"dino_2_optuna_05242325",
"dino_2_optuna_05242248",
"dino_2_optuna_05242231",
"dino_2_optuna_05242156",
"dino_2_optuna_05242055",
"dino_2_optuna_05242037",
"dino_2_optuna_05242019",]


In [19]:
valid_df = pd.read_parquet("../val2.pq")
valid_dataset = EmbeddingMetadataDataset(
    valid_df,
)
loader = DataLoader(valid_dataset, batch_size=128, shuffle=False)

poison = get_poison_mapping()

gt_df = pd.DataFrame()
gt_df['gt_class'] = valid_df['class_id'].apply(lambda x: x if x < 1604 else -1)
gt_df['gt_poison'] = gt_df['gt_class'].apply(lambda x : poison[x])


In [20]:
def score_model(model):
    preds = []
    
    for data in tqdm(loader):
        emb, metadata, label = data
        pred = model.predict(emb, metadata)
        preds.append(pred)
        
    preds = np.hstack(preds)
    
    preds_df = gt_df.copy()
    preds_df['preds'] = preds
    preds_df['preds_class'] = preds_df['preds'].apply(lambda x: x if x < 1604 else -1)
    preds_df['preds_poison'] = preds_df['preds_class'].apply(lambda x : poison[x])

    cls_report = classification_report(preds_df['gt_class'], preds_df['preds_class'], output_dict=True)
    cls_df = pd.DataFrame(cls_report).T

    poison_diff = (preds_df['preds_poison'] - preds_df['gt_poison']).apply(lambda x: 100 if x < 0 else x)
    acc = accuracy_score(preds_df['gt_class'], preds_df['preds_class'])
    track_1 = 1 - acc
    track_2 = poison_diff.sum() / len(poison_diff)
    track_3 = track_1 + track_2
    f1 = f1_score(preds_df['gt_class'], preds_df['preds_class'], average='macro')

    return dict(
        acc=acc,
        track_1=track_1,
        track_2=track_2,
        track_3=track_3,
        f1=f1,
        cls_df=cls_df,
    )

In [21]:
BASE_CKPT_PATH = "../checkpoints"
all_model_performance = []
models = {}

for mpath in model_paths:
    rpath = os.path.join(BASE_CKPT_PATH, mpath)
    for cpath in os.listdir(rpath):
        ckpt_path = os.path.join(rpath, cpath)
        print("evaluating ", ckpt_path)


        ckpt = torch.load(ckpt_path)
        model = FungiMEEModel()
        model.load_state_dict({w: ckpt['state_dict']["model." + w] for w in model.state_dict().keys()})
        model.eval()
        model.cuda()
        print("")

        model_name = mpath + "/" + cpath
        models[model_name] = model
        model_perf = {'name': model_name}
        model_perf.update(score_model(model))
        all_model_performance.append(model_perf)

evaluating  ../checkpoints/dino_2_optuna_05250105/epoch=12-step=33059.ckpt
Setting up Pytorch Model
Using devide: cuda:0



100%|██████████| 231/231 [00:01<00:00, 143.25it/s]
/home/chris/miniconda3/envs/fungiclef/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1497: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/home/chris/miniconda3/envs/fungiclef/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1497: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/home/chris/miniconda3/envs/fungiclef/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1497: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(av

evaluating  ../checkpoints/dino_2_optuna_05250105/epoch=13-step=35602.ckpt
Setting up Pytorch Model
Using devide: cuda:0



100%|██████████| 231/231 [00:01<00:00, 149.61it/s]
/home/chris/miniconda3/envs/fungiclef/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1497: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/home/chris/miniconda3/envs/fungiclef/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1497: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/home/chris/miniconda3/envs/fungiclef/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1497: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(av

evaluating  ../checkpoints/dino_2_optuna_05250105/epoch=14-step=38145.ckpt
Setting up Pytorch Model
Using devide: cuda:0



...

100%|██████████| 231/231 [00:01<00:00, 163.78it/s]
/home/chris/miniconda3/envs/fungiclef/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1497: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/home/chris/miniconda3/envs/fungiclef/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1497: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/home/chris/miniconda3/envs/fungiclef/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1497: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior

evaluating  ../checkpoints/dino_2_optuna_05242019/epoch=14-step=38145.ckpt
Setting up Pytorch Model
Using devide: cuda:0



100%|██████████| 231/231 [00:01<00:00, 161.60it/s]
/home/chris/miniconda3/envs/fungiclef/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1497: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/home/chris/miniconda3/envs/fungiclef/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1497: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/home/chris/miniconda3/envs/fungiclef/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1497: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior

evaluating  ../checkpoints/dino_2_optuna_05242019/epoch=9-step=25430.ckpt
Setting up Pytorch Model
Using devide: cuda:0



100%|██████████| 231/231 [00:01<00:00, 161.93it/s]
/home/chris/miniconda3/envs/fungiclef/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1497: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/home/chris/miniconda3/envs/fungiclef/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1497: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/home/chris/miniconda3/envs/fungiclef/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1497: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior

In [22]:
len(all_model_performance)

36

In [4]:
import timm
import torch

In [14]:
model = timm.create_model(
        "timm/vit_large_patch14_reg4_dinov2.lvd142m", pretrained=False
    )


In [15]:
weights = torch.load("../DSGT_FungiClef/checkpoints/dinov2.bin")

In [16]:
model.load_state_dict(weights)

<All keys matched successfully>

In [20]:
import os 

In [21]:
BASE_CKPT_PATH = "../DSGT_FungiClef/checkpoints"
ckpt_path = os.path.join(BASE_CKPT_PATH, "dino_2_optuna_05242055.ckpt")

In [30]:
os.listdir(BASE_CKPT_PATH)

['dino_2_optuna_05242231.ckpt',
 'dino_optuna_05241449.ckpt',
 'dino_optuna_05241257.ckpt',
 'dino_optuna_05241222.ckpt',
 'dino_2_optuna_05242055.ckpt',
 'dino_2_optuna_05242156.ckpt',
 'dino_2_optuna_05242344.ckpt',
 'dinov2.bin']

In [22]:
        ckpt = torch.load(ckpt_path)


In [28]:
        model = FungiMEEModel()


Setting up Pytorch Model
Using devide: cuda:0


In [29]:
        model.load_state_dict(
            {w: ckpt["model." + w] for w in model.state_dict().keys()}
        )

<All keys matched successfully>

In [23]:
perf = pd.DataFrame(all_model_performance)

In [24]:
perf.to_csv('model2.csv', index=False)

In [25]:
perf.sort_values(by='track_3')

,name,acc,track_1,track_2,track_3,f1,cls_df
13,dino_2_optuna_05242344/epoch=14-step=38145.ckpt,0.729821,0.270179,0.286975,0.557154,0.374001,precision recall f1-score ...
23,dino_2_optuna_05242231/epoch=9-step=25430.ckpt,0.722492,0.277508,0.282021,0.559529,0.404927,precision recall f1-score ...
28,dino_2_optuna_05242055/epoch=14-step=38145.ckpt,0.750416,0.249584,0.321684,0.571269,0.390723,precision recall f1-score ...
7,dino_2_optuna_05250025/epoch=13-step=35602.ckpt,0.734605,0.265395,0.306687,0.572083,0.371523,precision recall f1-score ...
21,dino_2_optuna_05242231/epoch=13-step=35602.ckpt,0.735928,0.264072,0.308859,0.572931,0.392060,precision recall f1-score ...
12,dino_2_optuna_05242344/epoch=13-step=35602.ckpt,0.723781,0.276219,0.310420,0.586639,0.377855,precision recall f1-score ...
35,dino_2_optuna_05242019/epoch=9-step=25430.ckpt,0.716215,0.283785,0.303328,0.587114,0.381760,precision recall f1-score ...
27,dino_2_optuna_05242055/epoch=13-step=35602.ckpt,0.737997,0.262003,0.331422,0.593424,0.383352,precision recall f1-score ...
22,dino_2_optuna_05242231/epoch=14-step=38145.ckpt,0.734774,0.265226,0.333492,0.598717,0.388910,precision recall f1-score ...
14,dino_2_optuna_05242344/epoch=9-step=25430.ckpt,0.724188,0.275812,0.323007,0.598819,0.385221,precision recall f1-score ...


In [16]:
perf.sort_values(by='track_3')

,name,acc,track_1,track_2,track_3,f1,cls_df
23,dino_optuna_05241222/epoch=9-step=25430.ckpt,0.731742,0.268258,0.373191,0.641449,0.394726,precision recall f1-score ...
20,dino_optuna_05241257/epoch=14-step=38145.ckpt,0.749106,0.250894,0.413673,0.664567,0.408910,precision recall f1-score ...
19,dino_optuna_05241257/epoch=13-step=35602.ckpt,0.742229,0.257771,0.417555,0.675326,0.406414,precision recall f1-score ...
22,dino_optuna_05241222/epoch=14-step=38145.ckpt,0.741309,0.258691,0.430084,0.688775,0.377716,precision recall f1-score ...
0,dino_05241908/epoch=19-step=50860.ckpt,0.738756,0.261244,0.432876,0.694120,0.373788,precision recall f1-score ...
2,dino_05241908/epoch=24-step=63575.ckpt,0.739028,0.260972,0.439583,0.700555,0.373442,precision recall f1-score ...
24,dino_optuna_05241449/epoch=12-step=33059.ckpt,0.742739,0.257261,0.446257,0.703517,0.376720,precision recall f1-score ...
9,dino_optuna_05241449/epoch=12-step=33059.ckpt,0.742739,0.257261,0.446257,0.703517,0.376720,precision recall f1-score ...
25,dino_optuna_05241449/epoch=13-step=35602.ckpt,0.746927,0.253073,0.462157,0.715229,0.373797,precision recall f1-score ...
10,dino_optuna_05241449/epoch=13-step=35602.ckpt,0.746927,0.253073,0.462157,0.715229,0.373797,precision recall f1-score ...


In [290]:
good_models = ["dino_2_optuna_05242055/epoch=14-step=38145.ckpt",
"dino_2_optuna_05242231/epoch=9-step=25430.ckpt",
"dino_2_optuna_05242344/epoch=14-step=38145.ckpt",
"dino_2_optuna_05242156/epoch=9-step=25430.ckpt",]

# for m in good_models:
#     shutil.copy(os.path.join("../checkpoints", m), os.path.join("../DSGT_FungiClef/checkpoints", m.split('/')[0]))

In [289]:
good_models = ["dino_optuna_05241257/epoch=14-step=38145.ckpt",
"dino_optuna_05241222/epoch=9-step=25430.ckpt", 
"dino_optuna_05241449/epoch=13-step=35602.ckpt"]

for m in good_models:
    shutil.copy(os.path.join("../checkpoints", m), os.path.join("../DSGT_FungiClef/checkpoints", m.split('/')[0]))

In [275]:
import shutil

In [181]:
torch.load("../checkpoints/dino_optuna_05241124/epoch=12-step=33059.ckpt")['state_dict']['model.head.fc2.weight'].shape


torch.Size([1719, 4096])

In [167]:
import torch.nn as nn

class FungiEnsembleModel(nn.Module):

    def __init__(self, models, softmax=True) -> None:
        super().__init__()

        self.models = nn.ModuleList()
        self.softmax = softmax
        self.device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

        for model in models:
            model = model.to(self.device)
            model.eval()
            self.models.append(model)
        
    def forward(self, img_emb, metadata):

        img_emb = img_emb.to(self.device)

        probs = []        

        for model in self.models:
            logits = model.forward(img_emb, metadata)
            
            p = logits.softmax(dim=1).detach().cpu() if self.softmax else logits.detach().cpu()

            probs.append(p)

        return torch.stack(probs).mean(dim=0)
    
    def predict(self, img_emb, metadata):
        
        logits = self.forward(img_emb, metadata)

        # Any preprocess happens here

        return logits.argmax(1).tolist()
    

In [291]:
ensemble_model = FungiEnsembleModel([models[g] for g in good_models])

In [295]:
all_model_performance.append(score_model(ensemble_model))

100%|██████████| 231/231 [00:04<00:00, 50.46it/s]
/home/chris/miniconda3/envs/fungiclef/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1497: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/home/chris/miniconda3/envs/fungiclef/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1497: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/home/chris/miniconda3/envs/fungiclef/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1497: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.

In [293]:
perf

,name,acc,track_1,track_2,track_3,f1,cls_df
0,dino_2_optuna_05250105/epoch=12-step=33059.ckpt,0.713908,0.286092,0.317443,0.603535,0.362531,precision recall f1-score ...
1,dino_2_optuna_05250105/epoch=13-step=35602.ckpt,0.716011,0.283989,0.367896,0.651885,0.364164,precision recall f1-score ...
2,dino_2_optuna_05250105/epoch=14-step=38145.ckpt,0.714383,0.285617,0.380280,0.665898,0.362100,precision recall f1-score ...
3,dino_2_optuna_05250045/epoch=12-step=33059.ckpt,0.736064,0.263936,0.386863,0.650799,0.374742,precision recall f1-score ...
4,dino_2_optuna_05250045/epoch=13-step=35602.ckpt,0.731653,0.268347,0.428087,0.696434,0.372897,precision recall f1-score ...
5,dino_2_optuna_05250045/epoch=14-step=38145.ckpt,0.732026,0.267974,0.360160,0.628134,0.375474,precision recall f1-score ...
6,dino_2_optuna_05250025/epoch=12-step=33059.ckpt,0.735521,0.264479,0.387779,0.652258,0.375745,precision recall f1-score ...
7,dino_2_optuna_05250025/epoch=13-step=35602.ckpt,0.734605,0.265395,0.306687,0.572083,0.371523,precision recall f1-score ...
8,dino_2_optuna_05250025/epoch=14-step=38145.ckpt,0.734096,0.265904,0.357242,0.623147,0.372344,precision recall f1-score ...
9,dino_2_optuna_05250006/epoch=12-step=33059.ckpt,0.736233,0.263767,0.363621,0.627388,0.380233,precision recall f1-score ...


In [292]:
ensemble_model.eval()

FungiEnsembleModel(
  (models): ModuleList(
    (0-3): 4 x FungiMEEModel(
      (date_embedding): MlpHead(
        (fc1): Linear(in_features=4, out_features=512, bias=True)
        (act): StarReLU(
          (relu): ReLU()
        )
        (norm): LayerNorm((512,), eps=1e-06, elementwise_affine=True)
        (fc2): Linear(in_features=512, out_features=1024, bias=True)
        (head_drop): Dropout(p=0.0, inplace=False)
      )
      (geo_embedding): MlpHead(
        (fc1): Linear(in_features=7, out_features=896, bias=True)
        (act): StarReLU(
          (relu): ReLU()
        )
        (norm): LayerNorm((896,), eps=1e-06, elementwise_affine=True)
        (fc2): Linear(in_features=896, out_features=1024, bias=True)
        (head_drop): Dropout(p=0.0, inplace=False)
      )
      (substr_embedding): MlpHead(
        (fc1): Linear(in_features=73, out_features=584, bias=True)
        (act): StarReLU(
          (relu): ReLU()
        )
        (norm): LayerNorm((584,), eps=1e-06, elemen

In [229]:
preds = []
for data in tqdm(loader):
    emb, metadata, _ = data
    pred = ensemble_model.forward(emb, metadata)
    preds.append(pred)
    


  0%|          | 0/230 [00:00<?, ?it/s]

  0%|          | 0/230 [00:00<?, ?it/s]


In [198]:
all_preds = torch.vstack(preds).numpy()

preds_df = metadata_df[['observation_id', 'image_path']]
preds_df['preds'] = [i for i in all_preds]
preds_df = preds_df[['observation_id', 'preds']].groupby('observation_id').mean().reset_index()
preds_df['class_id'] = preds_df['preds'].apply(lambda x: x.argmax() if x.argmax() <= 1603 else -1)
preds_df[['observation_id', 'class_id']]

In [ ]:
all_preds = all_preds[:100]

In [251]:
preds_df = metadata_df[['observation_id', 'image_path']]
preds_df.loc['preds'] = [i for i in all_preds]
preds_df = preds_df[['observation_id', 'preds']]

/tmp/ipykernel_218369/2033787745.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  preds_df.loc[:, 'preds'] = [i for i in all_preds]


In [250]:
test_id = 3412506378

In [297]:
perf = pd.DataFrame(all_model_performance)
perf

,name,acc,track_1,track_2,track_3,f1,cls_df
0,dino_2_optuna_05250105/epoch=12-step=33059.ckpt,0.713908,0.286092,0.317443,0.603535,0.362531,precision recall f1-score ...
1,dino_2_optuna_05250105/epoch=13-step=35602.ckpt,0.716011,0.283989,0.367896,0.651885,0.364164,precision recall f1-score ...
2,dino_2_optuna_05250105/epoch=14-step=38145.ckpt,0.714383,0.285617,0.380280,0.665898,0.362100,precision recall f1-score ...
3,dino_2_optuna_05250045/epoch=12-step=33059.ckpt,0.736064,0.263936,0.386863,0.650799,0.374742,precision recall f1-score ...
4,dino_2_optuna_05250045/epoch=13-step=35602.ckpt,0.731653,0.268347,0.428087,0.696434,0.372897,precision recall f1-score ...
5,dino_2_optuna_05250045/epoch=14-step=38145.ckpt,0.732026,0.267974,0.360160,0.628134,0.375474,precision recall f1-score ...
6,dino_2_optuna_05250025/epoch=12-step=33059.ckpt,0.735521,0.264479,0.387779,0.652258,0.375745,precision recall f1-score ...
7,dino_2_optuna_05250025/epoch=13-step=35602.ckpt,0.734605,0.265395,0.306687,0.572083,0.371523,precision recall f1-score ...
8,dino_2_optuna_05250025/epoch=14-step=38145.ckpt,0.734096,0.265904,0.357242,0.623147,0.372344,precision recall f1-score ...
9,dino_2_optuna_05250006/epoch=12-step=33059.ckpt,0.736233,0.263767,0.363621,0.627388,0.380233,precision recall f1-score ...


In [256]:
pd.read_csv('../DSGT_FungiClef/submission.csv')

,observation_id,class_id
0,3008824377,356
1,3008828397,1270
2,3008830357,1184
3,3008830358,693
4,3008830361,685
...,...,...
977,3803327303,-1
978,3828286302,667
979,3865752303,-1
980,3962112306,-1


In [228]:
preds_df

,observation_id,preds
0,3412506378,"[5.4560533e-06, 2.5688829e-05, 3.5050758e-05, ..."
1,3412506378,"[0.00015051023, 2.1697468e-07, 4.1416985e-08, ..."
2,3068189347,"[6.4058266e-05, 4.3651344e-06, 1.5596001e-07, ..."
3,3327302337,"[6.696428e-08, 4.3171607e-07, 1.36722695e-08, ..."
4,3358349363,"[6.6989855e-06, 4.737976e-08, 1.06338746e-07, ..."
...,...,...
95,3340172401,"[1.9471619e-08, 3.9753565e-09, 4.573509e-09, 1..."
96,3406489305,"[3.2272933e-07, 3.0060928e-06, 7.602231e-09, 1..."
97,3126942337,"[1.6399534e-08, 3.94173e-09, 1.684847e-06, 3.0..."
98,3039822303,"[5.692648e-06, 7.624916e-05, 8.598625e-06, 2.2..."


/tmp/ipykernel_218369/1177293005.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  preds_df['preds'] = [i for i in all_preds]


,observation_id,class_id
0,3014631355,-1
1,3014643340,225
2,3014647349,400
3,3018969380,374
4,3024139306,515
...,...,...
93,3422339309,357
94,3424494320,1109
95,3424495323,130
96,3429085301,-1


In [170]:
results = score_model(ensemble_model)

  0%|          | 0/230 [00:00<?, ?it/s]

100%|██████████| 230/230 [00:05<00:00, 44.45it/s]
/home/chris/miniconda3/envs/fungiclef/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1497: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/home/chris/miniconda3/envs/fungiclef/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1497: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/home/chris/miniconda3/envs/fungiclef/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1497: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.

In [172]:
d = {'name': 'ensemble'}
d.update(results)

all_model_performance.append(d)

In [173]:
perf = pd.DataFrame(all_model_performance)

In [174]:
perf

,name,acc,track_1,track_2,track_3,f1,cls_df
0,ensemble,0.77951,0.22049,0.38521,0.605699,0.451102,precision recall f1-score ...


In [161]:
metadata_df

,observation_id,image_path,m0,m1,d0,d1,g0,g1,g2,g3,...,habitat_26,habitat_27,habitat_28,habitat_29,habitat_30,habitat_31,class_id,poisonous,unknown,embedding
0,3412506378,0-3412506378.JPG,-8.660254e-01,5.000000e-01,-0.988468,0.151428,0.03125,0.96875,0.09375,0.68750,...,0,0,0,0,0,0,136,0,0,"[-0.86887753, -0.47589654, -0.35115266, 0.6738..."
1,3334610392,0-3334610392.JPG,-5.000000e-01,-8.660254e-01,0.299363,-0.954139,0.09375,0.31250,0.18750,0.53125,...,0,0,0,0,0,0,995,0,0,"[0.28158978, -0.5587861, -0.13272668, -0.12839..."
2,3068189347,0-3068189347.JPG,1.000000e+00,6.123234e-17,0.201299,0.979530,0.09375,0.31250,0.81250,0.25000,...,0,0,0,0,0,0,1121,0,0,"[-0.35207376, 0.5429209, -0.021078156, -0.4038..."
3,3327302337,1-3327302337.JPG,1.224647e-16,-1.000000e+00,-0.897805,-0.440394,0.03125,0.93750,0.96875,0.15625,...,0,0,0,0,0,0,107,0,0,"[0.4717792, -0.77035826, -0.6721257, 0.5593787..."
4,3358349363,0-3358349363.JPG,-8.660254e-01,-5.000000e-01,-0.790776,-0.612106,0.03125,0.96875,0.71875,0.06250,...,0,0,0,0,0,0,79,0,0,"[0.73937327, 0.55806947, -0.3651834, 0.4360694..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
29366,3329724347,0-3329724347.JPG,1.224647e-16,-1.000000e+00,-0.201299,0.979530,0.09375,0.25000,0.71875,0.37500,...,0,0,0,0,0,0,931,0,0,"[-0.419278, 0.66633576, -0.14421214, -1.171455..."
29367,3386484353,1-3386484353.JPG,-1.000000e+00,-1.836970e-16,0.790776,-0.612106,0.09375,0.31250,0.93750,0.06250,...,0,0,0,0,0,0,354,0,0,"[0.20411424, 0.07578691, 0.085911274, 0.406758..."
29368,3358360476,0-3358360476.JPG,-8.660254e-01,-5.000000e-01,-0.937752,0.347305,0.03125,0.93750,0.37500,0.62500,...,0,0,0,0,0,0,1047,0,0,"[-0.56015354, -0.18738864, -0.25037435, -0.242..."
29369,3313857314,0-3313857314.JPG,5.000000e-01,-8.660254e-01,-0.571268,0.820763,0.09375,0.25000,0.90625,0.81250,...,0,0,0,0,0,0,494,0,0,"[-0.24733329, 0.13334583, 1.0067952, -0.394908..."


In [107]:
BASE_CKPT_PATH = "../checkpoints"

models = {}

for mpath in model_paths:
    rpath = os.path.join(BASE_CKPT_PATH, mpath)
    for cpath in os.listdir(rpath):
        ckpt_path = os.path.join(rpath, cpath)
        print("evaluating ", ckpt_path)


        ckpt = torch.load(ckpt_path)
        model = FungiMEEModel()
        model.load_state_dict({w: ckpt['state_dict']["model." + w] for w in model.state_dict().keys()})
        model.eval()
        model.cuda()

        
        models[mpath + cpath] = model

evaluating  ../checkpoints/dino_2_optuna_05242055/epoch=13-step=35602.ckpt
Setting up Pytorch Model
Using devide: cuda:0
evaluating  ../checkpoints/dino_2_optuna_05242055/epoch=14-step=38145.ckpt
Setting up Pytorch Model
Using devide: cuda:0
evaluating  ../checkpoints/dino_2_optuna_05242055/epoch=9-step=25430.ckpt
Setting up Pytorch Model
Using devide: cuda:0
evaluating  ../checkpoints/dino_2_optuna_05242037/epoch=13-step=35602.ckpt
Setting up Pytorch Model
Using devide: cuda:0
evaluating  ../checkpoints/dino_2_optuna_05242037/epoch=14-step=38145.ckpt
Setting up Pytorch Model
Using devide: cuda:0
evaluating  ../checkpoints/dino_2_optuna_05242037/epoch=9-step=25430.ckpt
Setting up Pytorch Model
Using devide: cuda:0
evaluating  ../checkpoints/dino_2_optuna_05242019/epoch=8-step=22887.ckpt
Setting up Pytorch Model
Using devide: cuda:0
evaluating  ../checkpoints/dino_2_optuna_05242019/epoch=14-step=38145.ckpt
Setting up Pytorch Model
Using devide: cuda:0
evaluating  ../checkpoints/dino_2_o

In [59]:
models = []
for mpath in model_paths:
    rpath = os.path.join(BASE_CKPT_PATH, mpath)
    for cpath in os.listdir(rpath):
        print("evaluating ", cpath)
        ckpt_path = os.path.join(rpath, cpath)

        ckpt = torch.load(ckpt_path)
        model = FungiMEEModel()
        model.load_state_dict({w: ckpt['state_dict']["model." + w] for w in model.state_dict().keys()})
        model.eval()

        models.append(model)
    
    break

evaluating  epoch=13-step=35602.ckpt
Setting up Pytorch Model
Using devide: cuda:0
evaluating  epoch=14-step=38145.ckpt
Setting up Pytorch Model
Using devide: cuda:0
evaluating  epoch=9-step=25430.ckpt
Setting up Pytorch Model
Using devide: cuda:0


In [71]:
ensemble_model = FungiEnsembleModel(models)

In [72]:
img_emb, metadata, _ = next(iter(loader))

ensemble_model.forward(img_emb, metadata)

tensor([[1.6952e-09, 3.1039e-08, 2.7686e-07,  ..., 1.1336e-09, 1.0856e-08,
         5.5013e-03],
        [7.7530e-08, 1.2018e-08, 4.4817e-07,  ..., 1.0751e-08, 7.7453e-10,
         2.6579e-04],
        [1.0832e-05, 6.8688e-06, 1.3065e-07,  ..., 2.0353e-07, 1.8570e-08,
         6.7901e-05],
        ...,
        [1.1687e-06, 1.6865e-08, 3.1768e-08,  ..., 2.0692e-08, 1.3862e-09,
         1.4832e-03],
        [5.6377e-08, 1.3956e-08, 4.3703e-07,  ..., 1.2246e-06, 1.6330e-07,
         7.9367e-04],
        [2.9893e-09, 2.6990e-11, 2.0461e-11,  ..., 1.0303e-09, 5.6494e-10,
         7.3689e-07]])

In [18]:
from fungiclef.transforms import get_transforms
import pandas as pd

import torch

import timm

from PIL import Image
import os

import cv2

from tqdm import tqdm
from fungiclef.dataset import ImageMetadataDataset
import numpy as np
from torch.utils.data import DataLoader, Dataset

train_df = pd.read_parquet("../train.pq")
val_df = pd.read_parquet("../val.pq")
_df = pd.concat((train_df, val_df))

DIM = 518
BASE_PATH = "../data/DF_FULL"

transforms = get_transforms(data="valid", width=DIM, height=DIM)

valid_dataset = ImageMetadataDataset(
    _df, local_filepath="../data/DF_FULL/", transform=transforms)


In [14]:
transforms_2 = T.Compose([T.Resize((518, 518)),
                                     T.ToTensor(),
                                     T.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5])])

In [55]:
valid_dataset_2 = Ree(
    _df, local_filepath="../data/DF_FULL/", transform=transforms_2)

In [46]:
for d in valid_dataset_2:
    print(d)

<class 'numpy.ndarray'>


In [18]:
for data in valid_dataset:
    emb, metadata, _ = data

In [43]:
for data in loader:
    emb, metadata, _ = data
    break

In [31]:
ps = []

for _ in range(2):
    logits = model.forward(emb, metadata)
    ps.append(logits.softmax(dim=1).detach().cpu())

In [44]:
for _ in range(2):
    logits = model.forward(emb, metadata)
    ps.append(logits.softmax(dim=1).detach().cpu())

In [46]:
ps[3][0]

tensor([7.9697e-07, 1.1401e-06, 1.8302e-06,  ..., 4.0392e-09, 1.4066e-08,
        1.2006e-03])

In [41]:
ps[1][0]

tensor([1.5822e-05, 4.9770e-05, 5.4312e-07,  ..., 1.8243e-08, 3.7525e-08,
        5.2733e-05])

In [47]:
torch.stack(ps).mean(dim=0)[0]

RuntimeError: stack expects each tensor to be equal size, but got [59, 1717] at entry 0 and [128, 1717] at entry 2

In [16]:
preds = []
for data in tqdm(loader):
    emb, metadata, label = data
    pred = model.predict(emb, metadata)
    preds.append(pred)

100%|██████████| 230/230 [00:01<00:00, 131.01it/s]


In [24]:
results = score_model(model)

100%|██████████| 230/230 [00:01<00:00, 128.35it/s]
/home/chris/miniconda3/envs/fungiclef/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1497: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/home/chris/miniconda3/envs/fungiclef/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1497: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/home/chris/miniconda3/envs/fungiclef/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1497: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior

In [25]:
results

{'acc': 0.7413094549044976,
 'track_1': 0.2586905450955024,
 'track_2': 0.43008409655782914,
 'track_3': 0.6887746416533316,
 'f1': 0.3777156194583868,
 'cls_df':               precision    recall  f1-score       support
 -1             0.932511  0.669868  0.779665  10726.000000
 0              0.857143  0.545455  0.666667     22.000000
 1              0.000000  0.000000  0.000000      0.000000
 2              1.000000  0.800000  0.888889      5.000000
 3              0.500000  1.000000  0.666667      5.000000
 ...                 ...       ...       ...           ...
 1602           0.703704  0.558824  0.622951     34.000000
 1603           0.756098  0.584906  0.659574    159.000000
 accuracy       0.741309  0.741309  0.741309      0.741309
 macro avg      0.369171  0.402987  0.377716  29371.000000
 weighted avg   0.817829  0.741309  0.766936  29371.000000
 
 [1333 rows x 4 columns]}

In [234]:
accuracy_score(preds_df['gt_poison'], preds_df['preds_poison'])

0.977767185318852

In [235]:
preds_df['gt_poison'] == preds_df['preds_poison']

0        True
1        True
2        True
3        True
4        True
         ... 
29366    True
29367    True
29368    True
29369    True
29370    True
Length: 29371, dtype: bool

In [239]:
models = dino_2_optuna_05242055
dino_2_optuna_05242037
dino_2_optuna_05242019
dino_05241908
dino_05241702
dino_optuna_05241507
dino_optuna_05241449
dino_optuna_05241431
dino_optuna_05241354
dino_optuna_05241257
dino_optuna_05241222


0.43008409655782914

In [101]:
np.array(logits.squeeze().detach().cpu()).min()

-21.625269

In [50]:
np.array(logits).argmax(1)

array([1324])

In [43]:
emb.shape

torch.Size([1024])